# Checking the DQC Agent

In [ ]:
PATH = '../../scratch/aorl2/2026-04-08-00/2026-04-08-00.b7bf8a914965d2ce2cdfd7704faa38b5fee704b874bc391d21e3b9137701759c/'

CKPT_NUM = 900000

In [ ]:
import json
import os

import numpy as np

from agents import agents
from utils.datasets import Dataset, GCDataset, HGCDataset, CGCDataset
from utils.flax_utils import restore_agent

In [ ]:
flags_path = os.path.join(PATH, 'flags.json')
with open(flags_path, 'r') as f:
    saved_flags = json.load(f)

agent_config = saved_flags['agent']
dataset_class_name = agent_config.get('dataset_class', 'GCDataset')
dataset_class = {
    'GCDataset': GCDataset,
    'HGCDataset': HGCDataset,
    'CGCDataset': CGCDataset,
}[dataset_class_name]

dataset_path = os.path.join(PATH, 'data-100000.npz')
dataset_npz = np.load(dataset_path)
train_dataset = dataset_class(Dataset.create(**dict(dataset_npz)), config=agent_config)

seed = saved_flags.get('seed', 0)
example_batch = train_dataset.sample(1)

first_agent = agents[agent_config['agent_name']].create(seed, example_batch, agent_config)
first_agent = restore_agent(first_agent, PATH, CKPT_NUM)

print(f'Restored first_agent from checkpoint {CKPT_NUM}')


In [ ]:
agent = first_agent

In [ ]:
print(saved_flags['agent'])

In [ ]:
from tqdm import tqdm

all_cells = {}

for ob in tqdm(train_dataset.dataset['observations']):
    key = (np.floor(ob[0]), np.floor(ob[1]))
    if key in all_cells:
        all_cells[key] += 1
    else:
        all_cells[key] = 1

all_cell_points = np.asarray(list(all_cells.keys()))

## Checking Value Function

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], c='gray', alpha=0.1, s=10)
plt.show()

In [ ]:
default_obs = train_dataset.dataset['observations'][np.random.randint(0, 100000)]
all_cell_obs = np.repeat(default_obs[None], len(all_cell_points), axis=0)

for i in range(len(all_cell_points)):
    all_cell_obs[i, 0] = all_cell_points[i, 0]
    all_cell_obs[i, 1] = all_cell_points[i, 1]

goal = train_dataset.dataset['observations'][np.random.randint(0, 100000), :2]
all_goals = np.repeat(goal[None], len(all_cell_points), axis=0)

vs = agent.network.select('value')(observations=all_cell_obs, goals=all_goals)

plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], c=vs, cmap='viridis', alpha=1.0, s=68, marker='s')
plt.scatter(x=goal[0], y=goal[1], c='red', marker='x')
plt.show()

## Checking Subgoal Proposal

In [ ]:
import jax

rng = jax.random.PRNGKey(saved_flags['seed'])

In [ ]:
observation = default_obs
# goal = train_dataset.dataset['observations'][np.random.randint(100)][:2]
goal = np.asarray([37, 1])

rng, subgoal_rng = jax.random.split(rng)
subgoals = agent.propose_goals(observations=np.repeat(observation[None], 128, axis=0), goals=np.repeat(goal[None], 128, axis=0), rng=rng)

subgoal_obs = np.repeat(observation[None], 128, axis=0).copy()
for i in range(len(subgoals)):
    subgoal_obs[i, 0] = subgoals[i, 0]
    subgoal_obs[i, 1] = subgoals[i, 1]

vs = agent.network.select('value')(observations=subgoal_obs, goals=np.repeat(goal[None], 128, axis=0))

plt.figure(figsize=(20, 20))
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], c='gray', alpha=0.1, s=10)
plt.scatter(x=default_obs[0], y=default_obs[1], c='red', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', s=10)
b = plt.scatter(x=subgoals[..., 0], y=subgoals[..., 1], c=vs, cmap='viridis', s=5)
plt.colorbar(b)
plt.show()

In [ ]:
vs = agent.network.select('value')(observations=np.repeat(observation[None], 128, axis=0), goals=subgoals)

plt.figure(figsize=(20, 20))
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], c='gray', alpha=0.1, s=10)
plt.scatter(x=default_obs[0], y=default_obs[1], c='red', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', s=10)
b = plt.scatter(x=subgoals[..., 0], y=subgoals[..., 1], c=vs, cmap='viridis', s=5)
plt.colorbar(b)
plt.show()

## Check: is it goal-conditioned?

In [ ]:
replay_buffer = []


# ob = np.asarray([15.0, 20.0])
ob = train_dataset.dataset['observations'][1000].copy()
ob[0] = 15.0
ob[1] = 20.0
goal = np.asarray([20.0, 0.0])
rng = jax.random.PRNGKey(saved_flags['seed'])

for s in range(200):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoal = np.asarray(agent.propose_goals(ob, goal, sample_rng))
    print(subgoal)

    if np.linalg.norm(subgoal - ob[:2]) < 0.0025:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        # ob = subgoal
        ob[0] = subgoal[0].copy()
        ob[1] = subgoal[1].copy()

replay_buffer = np.asarray(replay_buffer)
# plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
# plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
print(len(replay_buffer))
print(replay_buffer)

## Filtering by Value

In [ ]:
replay_buffer = []


# ob = np.asarray([15.0, 20.0])
ob = train_dataset.dataset['observations'][1000].copy()
ob[0] = 15.0
ob[1] = 20.0
goal = np.asarray([20.0, 0.0])
rng = jax.random.PRNGKey(saved_flags['seed'])

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    
    subgoals = np.asarray(agent.propose_goals(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))
    # print(subgoal)

    subgoal_obs = np.repeat(ob[None], 128, axis=0)
    subgoal_obs[:, :2] = subgoals.copy()
    vs = agent.network.select('value')(subgoal_obs, np.repeat(goal[None], 128, axis=0))
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - ob[:2]) < 0.0025:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        # ob = subgoal
        ob[0] = subgoal[0].copy()
        ob[1] = subgoal[1].copy()

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
type(train_dataset)

In [ ]:
# batch = train_dataset.dataset.sample_sequence(batch_size=1, sequence_length=1, discount=0.995)
# print(batch['low_actor_goals'])

## Using the flow network instead

In [ ]:
saved_flags['env_name']

In [ ]:
config = dict(
    env_name='humanoidmaze-medium-navigate-v0',
    # dataset_path='../../scratch/aorl2/YOUR_RUN_DIR/data-1000000.npz',
    dataset_path='../../scratch/data/humanoidmaze-large-navigate-v0/humanoidmaze-large-navigate-v0seed-0.npz',
    observations_key='oracle_reps', # 'observations',
    goal_key='actor_goals',
    actions_key='low_actor_goals', #'actions',
    hidden_dims=(256, 256, 256),
    layer_norm=True,
    lr=3e-4,
    batch_size=256,
    num_train_steps=10000,
    log_interval=100,
    seed=0,
    value_p_curgoal=0.0,
    value_p_trajgoal=1.0,
    value_p_randomgoal=0.0,
    value_geom_sample=False,
    actor_p_curgoal=0.0,
    actor_p_trajgoal=1.0,
    actor_p_randomgoal=0.0,
    actor_geom_sample=True,
    gc_negative=False,
    subgoal_steps=25,
    discount=0.995,
    flow_steps=10,
    backup_horizon=25,
    goal_conditioned=True,
)

config

In [ ]:
from wrappers.datafuncs_utils import make_env_and_datasets

env, base_train_dataset, val_dataset = make_env_and_datasets(
    config['env_name'],
    dataset_path=config['dataset_path'],
    use_oracle_reps=True,
)
train_dataset = CGCDataset(base_train_dataset, config=config)

In [ ]:
from utils.networks import ActorVectorField

In [ ]:
from __future__ import annotations

from typing import Any

import flax
import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax

from utils.datasets import GCDataset
from utils.flax_utils import TrainState, nonpytree_field
from utils.networks import ActorVectorField, MLP
from wrappers.datafuncs_utils import make_env_and_datasets


In [ ]:
class GCFlowGoalProposerAgent(flax.struct.PyTreeNode):
    rng: Any
    network: TrainState
    config: Any = nonpytree_field()

    def flow_loss(self, batch, grad_params=None, rng=None):
        observations = batch[self.config['observations_key']]
        goals = batch[self.config['goal_key']] if self.config['goal_conditioned'] else None
        target_actions = batch[self.config['actions_key']]

        batch_size, action_dim = target_actions.shape
        rng = self.rng if rng is None else rng
        x_rng, t_rng = jax.random.split(rng)

        x_0 = jax.random.normal(x_rng, (batch_size, action_dim))
        t = jax.random.uniform(t_rng, (batch_size, 1))
        x_t = (1.0 - t) * x_0 + t * target_actions
        vel = target_actions - x_0

        pred_vel = self.network(
            observations,
            goals=goals,
            actions=x_t,
            times=t,
            params=grad_params,
        )
        loss = jnp.mean(jnp.square(pred_vel - vel))
        mae = jnp.mean(jnp.abs(pred_vel - vel))
        return loss, {
            'flow_loss': loss,
            'velocity_mae': mae,
        }

    @jax.jit
    def update(self, batch):
        new_rng, rng = jax.random.split(self.rng)

        def loss_fn(grad_params):
            return self.flow_loss(batch, grad_params, rng=rng)

        new_network, info = self.network.apply_loss_fn(loss_fn)
        info['step'] = new_network.step
        return self.replace(rng=new_rng, network=new_network), info

    @jax.jit
    def sample_actions(self, observations, goals, rng):
        single_example = observations.ndim == 1
        if not self.config['goal_conditioned']:
            goals = None
        if single_example:
            observations = observations[None, ...]
            if goals is not None:
                goals = goals[None, ...]

        x = jax.random.normal(rng, (observations.shape[0], self.config['action_dim']))

        for i in range(self.config['flow_steps']):
            t = jnp.full((observations.shape[0], 1), i / self.config['flow_steps'])
            vels = self.network(observations, goals=goals, actions=x, times=t)
            x = x + vels / self.config['flow_steps']

        return x[0] if single_example else x

    @classmethod
    def create(cls, example_batch, config):
        config = dict(config)
        config.setdefault('goal_conditioned', True)
        rng = jax.random.PRNGKey(config['seed'])
        rng, init_rng = jax.random.split(rng)
        action_dim = example_batch[config['actions_key']].shape[-1]
        model = ActorVectorField(
            hidden_dims=tuple(config['hidden_dims']),
            action_dim=action_dim,
            layer_norm=config['layer_norm'],
        )
        init_goals = example_batch[config['goal_key']] if config['goal_conditioned'] else None
        params = model.init(
            init_rng,
            example_batch[config['observations_key']],
            goals=init_goals,
            actions=example_batch[config['actions_key']],
            times=example_batch[config['actions_key']][..., :1],
        )['params']
        network = TrainState.create(model, params, tx=optax.adam(config['lr']))
        config['action_dim'] = action_dim
        return cls(rng=rng, network=network, config=flax.core.FrozenDict(config))

In [ ]:
example_batch = train_dataset.sample(1)

In [ ]:
example_batch.keys()

In [ ]:
flow_agent = GCFlowGoalProposerAgent.create(example_batch, config)
jax.tree_util.tree_map(lambda x: x.shape, flow_agent.network.params)

In [ ]:
flow_loss_history = []
velocity_mae_history = []

for step in range(1, config['num_train_steps'] + 1):
    batch = train_dataset.sample(config['batch_size'])
    flow_agent, info = flow_agent.update(batch)

    flow_loss_history.append(float(info['flow_loss']))
    velocity_mae_history.append(float(info['velocity_mae']))

    if step == 1 or step % config['log_interval'] == 0:
        print(
            f"step={step:05d} flow_loss={flow_loss_history[-1]:.6f} velocity_mae={velocity_mae_history[-1]:.6f}"
        )


In [ ]:
replay_buffer = []
# ob = np.asarray([20.0, 25.0])
# ob = np.asarray([0.0, 8.0])
# ob = np.asarray([5.0, 0.0])
# goal = np.asarray([20.0, 0.0])

# ob = np.asarray([15.0, 20.0])
ob = np.asarray([10.0, 0.0])
goal = np.asarray([20.0, 0.0])
goal = np.asarray([37, 1])
rng = jax.random.PRNGKey(config['seed'])

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoal = np.asarray(flow_agent.sample_actions(ob, goal, sample_rng))

    if np.linalg.norm(subgoal - ob) < 0.005:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

## Training a non- goal-conditioned goal proposer

In [ ]:
config = dict(
    env_name='humanoidmaze-medium-navigate-v0',
    # dataset_path='../../scratch/aorl2/YOUR_RUN_DIR/data-1000000.npz',
    dataset_path='../../scratch/data/humanoidmaze-large-navigate-v0/humanoidmaze-large-navigate-v0seed-0.npz',
    observations_key='oracle_reps', # 'observations',
    goal_key='actor_goals',
    actions_key='low_actor_goals', #'actions',
    hidden_dims=(256, 256, 256),
    layer_norm=True,
    lr=3e-4,
    batch_size=256,
    num_train_steps=100000,
    log_interval=100,
    seed=0,
    value_p_curgoal=0.0,
    value_p_trajgoal=1.0,
    value_p_randomgoal=0.0,
    value_geom_sample=False,
    actor_p_curgoal=0.0,
    actor_p_trajgoal=1.0,
    actor_p_randomgoal=0.0,
    actor_geom_sample=True,
    gc_negative=False,
    subgoal_steps=25,
    discount=0.995,
    flow_steps=10,
    backup_horizon=25,
    goal_conditioned=False,
)

config

In [ ]:
flow_agent = GCFlowGoalProposerAgent.create(example_batch, config)
jax.tree_util.tree_map(lambda x: x.shape, flow_agent.network.params)

flow_loss_history = []
velocity_mae_history = []

for step in range(1, config['num_train_steps'] + 1):
    batch = train_dataset.sample(config['batch_size'])
    flow_agent, info = flow_agent.update(batch)

    flow_loss_history.append(float(info['flow_loss']))
    velocity_mae_history.append(float(info['velocity_mae']))

    if step == 1 or step % config['log_interval'] == 0:
        print(
            f"step={step:05d} flow_loss={flow_loss_history[-1]:.6f} velocity_mae={velocity_mae_history[-1]:.6f}"
        )


In [ ]:
replay_buffer = []
# ob = np.asarray([20.0, 25.0])
# ob = np.asarray([0.0, 8.0])
# ob = np.asarray([5.0, 0.0])
# goal = np.asarray([20.0, 0.0])

# ob = np.asarray([15.0, 20.0])
ob = np.asarray([10.0, 0.0])
goal = np.asarray([20.0, 0.0])
goal = np.asarray([10.0, 8.0])
rng = jax.random.PRNGKey(config['seed'])

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoal = np.asarray(flow_agent.sample_actions(ob, goal, sample_rng))

    if np.linalg.norm(subgoal - ob) < 0.005:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
replay_buffer = []
# ob = np.asarray([20.0, 25.0])
# ob = np.asarray([0.0, 8.0])
# ob = np.asarray([5.0, 0.0])
# goal = np.asarray([20.0, 0.0])

# ob = np.asarray([15.0, 20.0])
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 0.0])
goal = np.asarray([10.0, 8.0])
rng = jax.random.PRNGKey(config['seed'])

for s in tqdm(range(600)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoal = np.asarray(flow_agent.sample_actions(ob, goal, sample_rng))

    if np.linalg.norm(subgoal - ob) < 0.005:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
# plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

Okay, confirmed that this doesn't seem goal-conditioned!

In [ ]:
# observation = default_obs
default_obs = train_dataset.dataset['observations'][np.random.randint(100)]
start = np.asarray([20.0, 08.0])
goal = np.asarray([37, 1])

ob = start.copy()

rng, subgoal_rng = jax.random.split(rng)
subgoals = flow_agent.sample_actions(observations=np.repeat(ob[None], 128, axis=0), goals=np.repeat(goal[None], 128, axis=0), rng=rng)
# np.asarray(flow_agent.sample_actions(ob, goal, sample_rng))

subgoal_obs = np.repeat(default_obs[None], 128, axis=0).copy()
for i in range(len(subgoals)):
    subgoal_obs[i, 0] = subgoals[i, 0]
    subgoal_obs[i, 1] = subgoals[i, 1]

vs = agent.network.select('value')(observations=subgoal_obs, goals=np.repeat(goal[None], 128, axis=0))

# plt.figure(figsize=(20, 20))
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], c='gray', alpha=0.1, s=10)
plt.scatter(x=start[0], y=start[1], c='red', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', s=10)
b = plt.scatter(x=subgoals[..., 0], y=subgoals[..., 1], c=vs, cmap='viridis', s=5, alpha=0.1)
plt.colorbar(b)
plt.show()

It's no longer unimodal!

In [ ]:
r = 1000000
low_actor_goals = []
actor_goals = []
samples = train_dataset.sample(batch_size=r, idxs=np.arange(r))

for i in tqdm(range(r)):
    sample_ob = samples['observations'][i]
    
    if np.linalg.norm(sample_ob[:2] - start) < 0.02:
        low_actor_goals.append(samples['low_actor_goals'][i])
        actor_goals.append(samples['actor_goals'][i])

low_actor_goals = np.asarray(low_actor_goals)
actor_goals = np.asarray(actor_goals)

if len(low_actor_goals) > 0:
    plt.figure(figsize=(8, 8))
    plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], c='gray', alpha=0.1, s=10)
    
    for i in range(len(low_actor_goals)):
        plt.plot(
            [low_actor_goals[i, 0], actor_goals[i, 0]],
            [low_actor_goals[i, 1], actor_goals[i, 1]],
            c='black',
            alpha=0.15,
            linewidth=0.5,
        )
    
    plt.scatter(x=low_actor_goals[..., 0], y=low_actor_goals[..., 1], s=10, label='low_actor_goals')
    plt.scatter(x=actor_goals[..., 0], y=actor_goals[..., 1], s=10, c='orange', label='actor_goals')
    plt.scatter(x=start[0], y=start[1], s=20, marker='x', c='red', label='start')
    plt.legend()
    plt.show()
else:
    print('not enough')


### Only argmax values

In [ ]:
replay_buffer = []
# ob = np.asarray([20.0, 25.0])
# ob = np.asarray([0.0, 8.0])
# ob = np.asarray([5.0, 0.0])
# goal = np.asarray([20.0, 0.0])

# ob = np.asarray([15.0, 20.0])
ob = np.asarray([10.0, 0.0])
goal = np.asarray([20.0, 0.0])
goal = np.asarray([37, 1])
rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    subgoals_obs[..., :2] = subgoals
    vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - goal) < 0.005:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
replay_buffer = []
# ob = np.asarray([20.0, 25.0])
# ob = np.asarray([0.0, 8.0])
# ob = np.asarray([5.0, 0.0])
# goal = np.asarray([20.0, 0.0])

# ob = np.asarray([15.0, 20.0])
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 0.0])
goal = np.asarray([37, 1])
rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    subgoals_obs[..., :2] = subgoals
    vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - goal) < 0.005:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

### Argmax sum of values

In [ ]:
replay_buffer = []
# ob = np.asarray([20.0, 25.0])
# ob = np.asarray([0.0, 8.0])
# ob = np.asarray([5.0, 0.0])
# goal = np.asarray([20.0, 0.0])

# ob = np.asarray([15.0, 20.0])
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 0.0])
goal = np.asarray([37, 1])
rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    subgoals_obs[..., :2] = subgoals

    ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - goal) < 0.005:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
ob = np.asarray([10.0, 0.0])
goal = np.asarray([20.0, 0.0])
goal = np.asarray([37, 1])

replay_buffer = []

rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    subgoals_obs[..., :2] = subgoals

    ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - goal) < 0.005:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
ob = np.asarray([10.0, 0.0])
# ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 15.0])

replay_buffer = []

rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    subgoals_obs[..., :2] = subgoals

    ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - goal) < 0.05:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

**Failure mode:** there can also be fixed points.

In [ ]:
# ob = np.asarray([10.0, 0.0])
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 15.0])

replay_buffer = []

rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    subgoals_obs[..., :2] = subgoals

    ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - goal) < 0.05:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
class GCMLPPolicy(nn.Module):
    hidden_dims: tuple[int, ...]
    action_dim: int
    layer_norm: bool = True

    @nn.compact
    def __call__(self, observations, goals):
        x = jnp.concatenate([observations, goals], axis=-1)
        x = MLP(self.hidden_dims, activate_final=True, layer_norm=self.layer_norm)(x)
        x = nn.Dense(self.action_dim)(x)
        return x


class GCILAgent(flax.struct.PyTreeNode):
    rng: Any
    network: TrainState
    config: Any = nonpytree_field()

    def actor_loss(self, batch, grad_params=None):
        observations = batch[self.config['observations_key']]
        goals = batch[self.config['goal_key']]
        pred_actions = self.network(observations, goals, params=grad_params)
        target_actions = batch[self.config['actions_key']]
        loss = jnp.mean(jnp.square(pred_actions - target_actions))
        mae = jnp.mean(jnp.abs(pred_actions - target_actions))
        return loss, {
            'loss': loss,
            'mae': mae,
        }

    @jax.jit
    def update(self, batch):
        new_rng, rng = jax.random.split(self.rng)

        def loss_fn(grad_params):
            return self.actor_loss(batch, grad_params)

        new_network, info = self.network.apply_loss_fn(loss_fn)
        info['step'] = new_network.step
        return self.replace(rng=new_rng, network=new_network), info

    @jax.jit
    def sample_actions(self, observations, goals):
        return self.network(observations, goals)

    @classmethod
    def create(cls, example_batch, config):
        rng = jax.random.PRNGKey(config['seed'])
        rng, init_rng = jax.random.split(rng)
        action_dim = example_batch[config['actions_key']].shape[-1]
        model = GCMLPPolicy(
            hidden_dims=tuple(config['hidden_dims']),
            action_dim=action_dim,
            layer_norm=config['layer_norm'],
        )
        params = model.init(
            init_rng,
            example_batch[config['observations_key']],
            example_batch[config['goal_key']],
        )['params']
        network = TrainState.create(model, params, tx=optax.adam(config['lr']))
        return cls(rng=rng, network=network, config=flax.core.FrozenDict(config))


In [ ]:
deterministic_agent = GCILAgent.create(example_batch, config)
jax.tree_util.tree_map(lambda x: x.shape, agent.network.params)

loss_history = []
mae_history = []

for step in range(1, config['num_train_steps'] + 1):
    batch = train_dataset.sample(config['batch_size'])
    deterministic_agent, info = deterministic_agent.update(batch)

    loss_history.append(float(info['loss']))
    mae_history.append(float(info['mae']))

    if step == 1 or step % config['log_interval'] == 0:
        print(
            f"step={step:05d} loss={loss_history[-1]:.6f} mae={mae_history[-1]:.6f}"
        )

plt.figure(figsize=(8, 4))
plt.plot(loss_history, label='mse loss')
plt.plot(mae_history, label='mae')
plt.xlabel('training step')
plt.ylabel('metric')
plt.title('Offline GC imitation learning')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
eval_batch = train_dataset.sample(8)
pred_actions = np.asarray(
    deterministic_agent.sample_actions(
        eval_batch[config['observations_key']],
        eval_batch[config['goal_key']],
    )
)
target_actions = np.asarray(eval_batch[config['actions_key']])

print('predicted actions shape:', pred_actions.shape)
print('target actions shape:', target_actions.shape)
print('sample mse:', np.mean((pred_actions - target_actions) ** 2))

In [ ]:
# ob = np.asarray([10.0, 0.0])
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 15.0])

replay_buffer = []

rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    # rng, sample_rng = jax.random.split(rng)
    # subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    # subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    # subgoals_obs[..., :2] = subgoals

    # ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    # subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    # vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    
    # idx = np.argmax(vs)
    # subgoal = subgoals[idx]
    subgoal = deterministic_agent.sample_actions(ob, goal)

    if np.linalg.norm(subgoal - goal) < 0.05:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Deterministic proposer rollout')
plt.show()

In [ ]:
# ob = np.asarray([10.0, 0.0])
ob = np.asarray([11.0, 15.0])
goal = np.asarray([37.0, 1.0])

replay_buffer = []

rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    # rng, sample_rng = jax.random.split(rng)
    # subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    # subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    # subgoals_obs[..., :2] = subgoals

    # ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    # subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    # vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    
    # idx = np.argmax(vs)
    # subgoal = subgoals[idx]
    subgoal = deterministic_agent.sample_actions(ob, goal)

    if np.linalg.norm(subgoal - goal) < 0.05:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Deterministic proposer rollout')
plt.show()

In [ ]:
# ob = np.asarray([10.0, 0.0])
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 24.0])

replay_buffer = []

rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    # rng, sample_rng = jax.random.split(rng)
    # subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    # subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    # subgoals_obs[..., :2] = subgoals

    # ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    # subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    # vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    
    # idx = np.argmax(vs)
    # subgoal = subgoals[idx]
    subgoal = deterministic_agent.sample_actions(ob, goal)

    if np.linalg.norm(subgoal - goal) < 0.05:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Deterministic proposer rollout')
plt.show()

In [ ]:
# ob = np.asarray([10.0, 0.0])
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 24.0])

replay_buffer = []

rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    subgoals_obs[..., :2] = subgoals

    # ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    # vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    vs = subgoal_to_goal_vs
    
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - goal) < 0.05:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 16.0])

replay_buffer = []
rng = jax.random.PRNGKey(config['seed'])

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    subgoal = deterministic_agent.sample_actions(ob, goal)

    if np.linalg.norm(subgoal - goal) < 0.05:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Deterministic proposer rollout')
plt.show()

In [ ]:
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 16.0])

replay_buffer = []

rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    subgoals_obs[..., :2] = subgoals

    # ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    # vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    vs = subgoal_to_goal_vs
    
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - goal) < 0.05:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
ob = np.asarray([11.0, 15.0])
goal = np.asarray([20.0, 16.0])

replay_buffer = []

rng = jax.random.PRNGKey(config['seed'])

# default_obs = 

for s in tqdm(range(200)):
    replay_buffer.append(ob)
    rng, sample_rng = jax.random.split(rng)
    subgoals = np.asarray(flow_agent.sample_actions(np.repeat(ob[None], 128, axis=0), np.repeat(goal[None], 128, axis=0), sample_rng))

    subgoals_obs = np.repeat(default_obs[None], 128, axis=0)
    subgoals_obs[..., :2] = subgoals

    ob_to_subgoal_vs = agent.network.select('value')(np.repeat(ob[None], 128, axis=0), subgoals_obs)
    subgoal_to_goal_vs = agent.network.select('value')(subgoals_obs, np.repeat(goal[None], 128, axis=0))

    vs = ob_to_subgoal_vs + subgoal_to_goal_vs
    # vs = subgoal_to_goal_vs
    
    idx = np.argmax(vs)
    subgoal = subgoals[idx]

    if np.linalg.norm(subgoal - goal) < 0.05:
        print(f'reached at step {s}!')
        print('subgoal:', subgoal)
        break
    else:
        ob = subgoal

replay_buffer = np.asarray(replay_buffer)
plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], s=10, alpha=0.1, c='gray')
plt.scatter(x=replay_buffer[..., 0], y=replay_buffer[..., 1], c=np.arange(len(replay_buffer)), cmap='viridis', s=10)
plt.scatter(x=goal[0], y=goal[1], c='green', marker='*')
plt.scatter(x=replay_buffer[0][0], y=replay_buffer[0][1], marker='x', c='red')
plt.title('Flow proposer rollout')
plt.show()

In [ ]:
end = np.asarray([20.0, 16.0])
start = np.asarray([11.0, 15.0])

subgoal_to_goal = agent.network.select('value')(observations=all_cell_obs, goals=np.repeat(end[None], len(all_cell_obs), axis=0))

ob_to_subgoal = agent.network.select('value')(observations=np.repeat(start[None], len(all_cell_obs), axis=0), goals=all_cell_obs)

# vs = np.log(subgoal_to_goal + ob_to_subgoal + 10.0)
# vs = subgoal_to_goal + ob_to_subgoal
vs = ob_to_subgoal

c = plt.scatter(x=all_cell_points[..., 0], y=all_cell_points[..., 1], c=vs, cmap='viridis', alpha=1.0, s=40, marker='s')
plt.scatter(x=start[0], y=start[1], c='red', marker='x')
plt.scatter(x=end[0], y=end[1], c='green')
plt.title('sum of value to and value from')
plt.colorbar(c)
plt.show()

In [ ]:
agent